In [51]:
import pandas as pd
import numpy as np
import os
import sys
import json
import time
import datetime


# reading & processing return report  for US region for now

In [52]:
from pathlib import Path

import pandas as pd


base_path = Path(
    r"C:\Users\Shivank Rajput\OneDrive - EcoSoul Home\Central Repository\revenue\us\input\amazon\fba_customer_returns"
)

# Columns to keep from source files
required_cols = [
    "return-date",
    "order-id",
    "sku",
    "quantity",
    "detailed-disposition",
    "reason",
    "customer-comments",
]

# Read TSV, TXT and CSV files recursively
allowed_extensions = {".tsv", ".txt", ".csv"}

files = [
    file
    for file in base_path.rglob("*")
    if file.is_file() and file.suffix.lower() in allowed_extensions
]

print(f"Total files found: {len(files)}")

dfs = []
skipped_files = []
failed_files = []

for file in files:

    # Only process files starting with fba_customer_returns_
    if not file.name.startswith("fba_customer_returns_"):
        print(f"SKIPPED: {file.name}")
        skipped_files.append(file)
        continue

    try:
        print(f"Reading: {file}")

        # Determine separator based on file extension
        if file.suffix.lower() == ".csv":
            sep = ","
        else:
            # TSV and TXT
            sep = "\t"

        # Read ONLY required columns
        df = pd.read_csv(
            file,
            sep=sep,
            encoding="cp1252",
            usecols=required_cols,
        )

        # Add source file information
        df["source_file"] = file.name

        dfs.append(df)

    except Exception as e:
        print(f"FAILED: {file}")
        print(f"ERROR: {e}")
        failed_files.append((file, str(e)))


# Combine all processed files
if dfs:
    returns_df = pd.concat(
        dfs,
        ignore_index=True,
    )
else:
    returns_df = pd.DataFrame(
        columns=required_cols + ["source_file"]
    )


print("\n========== SUMMARY ==========")
print(f"Total files found : {len(files)}")
print(f"Files processed   : {len(dfs)}")
print(f"Files skipped     : {len(skipped_files)}")
print(f"Files failed      : {len(failed_files)}")
print(f"Final rows        : {len(returns_df)}")
print(f"Final columns     : {returns_df.columns.tolist()}")

print("\n========== COLUMN CHECK ==========")
print(returns_df.columns.tolist())

Total files found: 68
Reading: C:\Users\Shivank Rajput\OneDrive - EcoSoul Home\Central Repository\revenue\us\input\amazon\fba_customer_returns\2026\apr\fba_customer_returns_Apr-2026.csv
Reading: C:\Users\Shivank Rajput\OneDrive - EcoSoul Home\Central Repository\revenue\us\input\amazon\fba_customer_returns\2026\aug\fba_customer_returns_Aug-2026.csv
Reading: C:\Users\Shivank Rajput\OneDrive - EcoSoul Home\Central Repository\revenue\us\input\amazon\fba_customer_returns\2026\feb\fba_customer_returns_Feb-2026.csv
Reading: C:\Users\Shivank Rajput\OneDrive - EcoSoul Home\Central Repository\revenue\us\input\amazon\fba_customer_returns\2026\jan\fba_customer_returns_Jan-2026.csv
Reading: C:\Users\Shivank Rajput\OneDrive - EcoSoul Home\Central Repository\revenue\us\input\amazon\fba_customer_returns\2026\jul\fba_customer_returns_Jul-2026.csv
Reading: C:\Users\Shivank Rajput\OneDrive - EcoSoul Home\Central Repository\revenue\us\input\amazon\fba_customer_returns\2026\jun\fba_customer_returns_Jun-202

In [53]:
returns_df

                     return-date             order-id            sku  \
0      2026-05-01T05:51:52+00:00  114-0217862-4192274      PLP6SQ100   
1      2026-05-01T05:03:33+00:00  111-3966579-5611457       PLB6RO50   
2      2026-05-01T05:02:18+00:00  111-3966579-5611457  PLP10RO50-New   
3      2026-05-01T04:06:27+00:00  113-4044389-4135467       PLP6RO50   
4      2026-05-01T03:35:37+00:00  114-7699108-4020259   PLP8SQ50-New   
...                          ...                  ...            ...   
39299  2021-09-02T04:05:31-07:00  112-6671189-9456227     PLP10RO020   
39300  2021-09-01T22:08:59-07:00  114-9249971-7747440       CTB8G100   
39301  2021-09-01T22:04:33-07:00  111-2654988-7766629       PLB5RO50   
39302  2021-09-01T20:16:21-07:00  111-3645045-8593029     PLP10RO020   
39303  2021-09-01T04:17:39-07:00  112-9865539-8842635      PLP10SQ50   

       quantity detailed-disposition                     reason  \
0             1             SELLABLE              UNWANTED_ITEM   
1

# using  return_directory.yaml to mapp against reason and customer comment cols

In [54]:
import html
import re
import unicodedata
from pathlib import Path

import yaml

# Project root: .../Centralized Ecosystem Portal
PROJECT_ROOT = Path.cwd()
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "config_manager" / "return_directory.yaml").exists():
        PROJECT_ROOT = _p
        break

RETURN_DIR_PATH = PROJECT_ROOT / "config_manager" / "return_directory.yaml"
with RETURN_DIR_PATH.open(encoding="utf-8") as _f:
    return_directory = yaml.safe_load(_f)

alerts_cfg = return_directory.get("alerts") or {}
reason_map = return_directory.get("return_reason_mapping") or {}
prefix_map = return_directory.get("structured_comment_prefixes") or {}
groups_cfg = return_directory.get("customer_comment_groups") or {}
cleaning_cfg = return_directory.get("cleaning") or {}

UNKNOWN_REASON_GROUP = alerts_cfg.get("unknown_reason_group", "Unmapped Reason")
UNKNOWN_COMMENT_GROUP = alerts_cfg.get("unknown_comment_group", "Unmapped")
ALERTS_ENABLED = bool(alerts_cfg.get("enabled", True))
ALERT_UNKNOWN_REASON = ALERTS_ENABLED and bool(alerts_cfg.get("on_unknown_reason", True))
ALERT_UNKNOWN_COMMENT = ALERTS_ENABLED and bool(alerts_cfg.get("on_unknown_comment", True))
COLLECT_UNKNOWNS = bool(alerts_cfg.get("collect_unknowns", True))

# group_key -> label (for output columns)
group_labels = {
    key: (cfg or {}).get("label", key)
    for key, cfg in groups_cfg.items()
}

# Build alias lookup (normalized comment -> group_key) and compiled keyword regexes
def _normalize_key(text: str) -> str:
    text = html.unescape(str(text))
    text = unicodedata.normalize(
        cleaning_cfg.get("unicode_normalize") or "NFKC", text
    )
    text = re.sub(r"\s+", " ", text).strip().lower()
    text = re.sub(r"[^\w\s|/.-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _clean_comment(text) -> str:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""
    text = html.unescape(str(text))
    text = unicodedata.normalize(
        cleaning_cfg.get("unicode_normalize") or "NFKC", text
    )
    if cleaning_cfg.get("collapse_whitespace", True):
        text = re.sub(r"\s+", " ", text).strip()
    return text


treat_as_empty = {
    _normalize_key(x) for x in (cleaning_cfg.get("treat_as_empty") or [])
}
treat_as_empty.update({"", "nan", "none", "null", "n/a", "na"})

alias_to_group: dict[str, str] = {}
keyword_rules: list[tuple[str, re.Pattern]] = []
for group_key, cfg in groups_cfg.items():
    if group_key == "unmapped":
        continue
    cfg = cfg or {}
    for alias in cfg.get("aliases") or []:
        nk = _normalize_key(alias)
        if nk and nk not in alias_to_group:
            alias_to_group[nk] = group_key
    for pat in cfg.get("keywords") or []:
        try:
            keyword_rules.append((group_key, re.compile(pat, re.I)))
        except re.error as exc:
            print(f"WARN bad keyword regex in {group_key}: {pat!r} ({exc})")

# prefix keys normalized for case-insensitive match
prefix_lookup = {_normalize_key(k): v for k, v in prefix_map.items()}

unknown_reasons: set[str] = set()
unknown_comments: set[str] = set()


def map_return_reason(reason) -> str:
    if reason is None or (isinstance(reason, float) and pd.isna(reason)):
        code = ""
    else:
        code = str(reason).strip()
    if not code:
        return UNKNOWN_REASON_GROUP
    group = reason_map.get(code)
    if group is None:
        if COLLECT_UNKNOWNS or ALERT_UNKNOWN_REASON:
            unknown_reasons.add(code)
        return UNKNOWN_REASON_GROUP
    return group


def map_customer_comment(comment) -> tuple[str, str]:
    """Return (group_key, group_label)."""
    cleaned = _clean_comment(comment)
    key = _normalize_key(cleaned)

    if key in treat_as_empty:
        gk = "noise_empty"
        return gk, group_labels.get(gk, gk)

    # 1) exact alias
    if key in alias_to_group:
        gk = alias_to_group[key]
        return gk, group_labels.get(gk, gk)

    # 2) structured Amazon prefix before first '|'
    if "|" in key:
        head = key.split("|", 1)[0].strip()
        if head in prefix_lookup:
            gk = prefix_lookup[head]
            # refine Incorrect Size -> too small / too large when possible
            if gk == "wrong_size_ordered":
                if re.search(r"too small|to small|smaller|tiny|snall", key):
                    gk = "size_too_small"
                elif re.search(r"too (large|big)|to big|larger|bigger", key):
                    gk = "size_too_large"
            if gk == "no_longer_needed_changed_mind" and re.search(
                r"late|arriv|deliver", key
            ):
                gk = "delivery_late"
            return gk, group_labels.get(gk, gk)

    # 3) keyword regexes (first match wins; YAML group order)
    for gk, cre in keyword_rules:
        if cre.search(key):
            return gk, group_labels.get(gk, gk)

    # 4) unmapped (+ collect for alert summary)
    gk = "unmapped"
    if COLLECT_UNKNOWNS or ALERT_UNKNOWN_COMMENT:
        unknown_comments.add(cleaned)
    return gk, group_labels.get(gk, UNKNOWN_COMMENT_GROUP)


# --- apply mappings ---
returns_df = returns_df.copy()
returns_df["return_reason_group"] = returns_df["reason"].map(map_return_reason)

_comment_mapped = returns_df["customer-comments"].map(map_customer_comment)
returns_df["customer_comment_group_key"] = _comment_mapped.map(lambda x: x[0])
returns_df["customer_comment_group"] = _comment_mapped.map(lambda x: x[1])

print(f"Loaded return directory: {RETURN_DIR_PATH}")
print(f"Reason groups mapped   : {returns_df['return_reason_group'].nunique()}")
print(f"Comment groups mapped  : {returns_df['customer_comment_group'].nunique()}")
print(f"Unknown reasons        : {len(unknown_reasons)}")
print(f"Unknown comments       : {len(unknown_comments)}")

if ALERT_UNKNOWN_REASON and unknown_reasons:
    print("\n=== ALERT unknown return reasons ===")
    for code in sorted(unknown_reasons):
        print(f"ALERT unknown return reason: {code}")

if ALERT_UNKNOWN_COMMENT and unknown_comments:
    print("\n=== ALERT unknown customer comments (unique) ===")
    for text in sorted(unknown_comments, key=str.lower):
        print(f"ALERT unknown customer comment: {text}")

returns_df

Loaded return directory: c:\Developer data\Centralized Ecosystem Portal\config_manager\return_directory.yaml
Reason groups mapped   : 10
Comment groups mapped  : 18
Unknown reasons        : 0
Unknown comments       : 1355

=== ALERT unknown customer comments (unique) ===
ALERT unknown customer comment: ***LEAD*** Concession Request > Valid Request > WMS > Undeliverable / Returning to Seller
ALERT unknown customer comment: **LEAD** Product does not meet expectations
ALERT unknown customer comment: *-lead* cannot be delivery
ALERT unknown customer comment: .Many of the plates looked as if they had been used
ALERT unknown customer comment: 1, 25 package of 10” square plates . No brand label on item.
ALERT unknown customer comment: 10” Square Plates
ALERT unknown customer comment: 4 inch
ALERT unknown customer comment: 8" plates sent not 10"
ALERT unknown customer comment: A case of bamboo utensil sets
ALERT unknown customer comment: a round plate with decorations
ALERT unknown customer co

                     return-date             order-id            sku  \
0      2026-05-01T05:51:52+00:00  114-0217862-4192274      PLP6SQ100   
1      2026-05-01T05:03:33+00:00  111-3966579-5611457       PLB6RO50   
2      2026-05-01T05:02:18+00:00  111-3966579-5611457  PLP10RO50-New   
3      2026-05-01T04:06:27+00:00  113-4044389-4135467       PLP6RO50   
4      2026-05-01T03:35:37+00:00  114-7699108-4020259   PLP8SQ50-New   
...                          ...                  ...            ...   
39299  2021-09-02T04:05:31-07:00  112-6671189-9456227     PLP10RO020   
39300  2021-09-01T22:08:59-07:00  114-9249971-7747440       CTB8G100   
39301  2021-09-01T22:04:33-07:00  111-2654988-7766629       PLB5RO50   
39302  2021-09-01T20:16:21-07:00  111-3645045-8593029     PLP10RO020   
39303  2021-09-01T04:17:39-07:00  112-9865539-8842635      PLP10SQ50   

       quantity detailed-disposition                     reason  \
0             1             SELLABLE              UNWANTED_ITEM   
1

In [55]:
returns_df['return_date'] = pd.to_datetime(returns_df['return-date'], utc=True).dt.tz_localize(None)
returns_df['year_month'] = returns_df['return_date'].dt.to_period('M').astype(str)
returns_df['return_hour'] = returns_df['return_date'].dt.hour
returns_df['return_time'] = returns_df['return_date'].dt.strftime("%H:%M")
returns_df['return_date'] = returns_df['return_date'].apply(lambda x: x.strftime("%Y-%m-%d") if pd.notnull(x) else None)

returns_df = returns_df.loc[:,['year_month', 'return_date','return_hour', 'return_time','order-id', 'sku', 'quantity', 'detailed-disposition',
                               'return_reason_group','customer_comment_group', 'source_file',]]

In [57]:
returns_df

      year_month return_date  return_hour return_time             order-id  \
0        2026-05  2026-05-01            5       05:51  114-0217862-4192274   
1        2026-05  2026-05-01            5       05:03  111-3966579-5611457   
2        2026-05  2026-05-01            5       05:02  111-3966579-5611457   
3        2026-05  2026-05-01            4       04:06  113-4044389-4135467   
4        2026-05  2026-05-01            3       03:35  114-7699108-4020259   
...          ...         ...          ...         ...                  ...   
39299    2021-09  2021-09-02           11       11:05  112-6671189-9456227   
39300    2021-09  2021-09-02            5       05:08  114-9249971-7747440   
39301    2021-09  2021-09-02            5       05:04  111-2654988-7766629   
39302    2021-09  2021-09-02            3       03:16  111-3645045-8593029   
39303    2021-09  2021-09-01           11       11:17  112-9865539-8842635   

                 sku  quantity detailed-disposition  \
0       

In [58]:
# SKU cleaning — same path as orders_report (esh_sku_cleaning + local COGS)
import sys
from pathlib import Path

# Resolve project root (works even if earlier cells were not re-run)
_PROJECT_ROOT = Path.cwd()
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "config_manager" / "esh_sku_cleaning.py").exists():
        _PROJECT_ROOT = _p
        break
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from config_manager import esh_sku_cleaning

GEO = "US"  # this notebook is US-only for now
CHANNEL = "amazon"

print(f">>> STEP: load SKU mapping (esh_sku_cleaning, geo={GEO}, channel={CHANNEL})")
sku_mapping = esh_sku_cleaning.load_sku_mapping(geos=[GEO], channel=CHANNEL)
print(f"SKU mapping entries: {len(sku_mapping):,}")

rows_before = len(returns_df)
print(f">>> STEP: clean sku column ({rows_before:,} rows)")
returns_df = esh_sku_cleaning.clean_sku_column(
    returns_df,
    sku_column="sku",
    sku_mapping=sku_mapping,
    drop_missing=True,
)
print(
    f"Rows after SKU clean: {len(returns_df):,} "
    f"(dropped {rows_before - len(returns_df):,})"
)
returns_df

>>> STEP: load SKU mapping (esh_sku_cleaning, geo=US, channel=amazon)
  Loaded local COGS C:\Developer data\Centralized Ecosystem Portal\cogs\esh\us_amazon_cogs.parquet: 476 rows
SKU mapping aliases loaded: 208 (from cogs/esh geos=['US'] channel=amazon)
SKU mapping entries: 208
>>> STEP: clean sku column (39,304 rows)
Rows after SKU clean: 39,304 (dropped 0)


      year_month return_date  return_hour return_time             order-id  \
0        2026-05  2026-05-01            5       05:51  114-0217862-4192274   
1        2026-05  2026-05-01            5       05:03  111-3966579-5611457   
2        2026-05  2026-05-01            5       05:02  111-3966579-5611457   
3        2026-05  2026-05-01            4       04:06  113-4044389-4135467   
4        2026-05  2026-05-01            3       03:35  114-7699108-4020259   
...          ...         ...          ...         ...                  ...   
39299    2021-09  2021-09-02           11       11:05  112-6671189-9456227   
39300    2021-09  2021-09-02            5       05:08  114-9249971-7747440   
39301    2021-09  2021-09-02            5       05:04  111-2654988-7766629   
39302    2021-09  2021-09-02            3       03:16  111-3645045-8593029   
39303    2021-09  2021-09-01           11       11:17  112-9865539-8842635   

             sku  quantity detailed-disposition          return

In [62]:
returns_df = pd.pivot_table(
    returns_df,
    index=[
        "year_month",
        "return_date",
        "return_hour",
        "order-id",
        "sku",
        "detailed-disposition",
        "return_reason_group",
        "customer_comment_group",
        "source_file",
    ],
    values="quantity",
    aggfunc="sum",
).reset_index()
returns_df

      year_month return_date  return_hour             order-id           sku  \
0        2021-03  2021-03-11            7  113-2292839-4574605      PLB5RO50   
1        2021-03  2021-03-20            1  111-0919331-3088263        BCS175   
2        2021-03  2021-03-22           22  112-9421001-3523426          BCB5   
3        2021-03  2021-03-23           10  111-3626253-3428256     CTBN4G100   
4        2021-03  2021-03-23           14  113-0052922-0473030     PLP10SQ25   
...          ...         ...          ...                  ...           ...   
35861    2026-10  2026-10-02           22  111-2009120-8285019        CZLS25   
35862    2026-10  2026-10-02           22  113-8445820-3978662   BFW48P04-AV   
35863    2026-10  2026-10-02           23  112-0488802-2489033      PLP8SQ50   
35864    2026-10  2026-10-03            0  702-0842996-8059447  PLPB7&10SQ50   
35865    2026-10  2026-10-03            2  701-9352874-1473011     CBP9RO100   

      detailed-disposition          ret

In [ ]:
# furthe condition
# need to create this script to run on all geo for esh/ knc
# save the final dataframe to local as well azure 